# 43008 Reinforcement Learning -- Assignment 3, Part E: Value-Based Deep Reinforcement Learning on Snake

**The Deadly Triad (project 48): Yue Liu (25211451), Jiahao Liu (14191136), Siqin Jin (26145324)**

**Project**: a controlled study of state representation, reward design and DQN stabilisation on a self-built Snake
environment (8 x 8 and 10 x 10 boards): tabular Q-learning; DQN on an 11-feature vector (observation A, MLP) or an
8-plane grid (observation B8, CNN); the discount factor; the replay buffer and the target network; Double DQN, Dueling
and prioritised replay; four reward designs (R0 sparse, R1 step penalty, R2 potential-based shaping, R3 naive distance
bonus) and a reward-hacking probe; random, BFS-greedy and Hamiltonian-cycle reference players. Every configuration is
trained with five seeds, the seed is the unit of replication and every interval is a 95 % Student-t interval over seeds.

**What this notebook does**: it regenerates every table and figure of the report from the training logs and shows them
in the report's order with the report's captions (Section 5), then checks every computed table against the report
source cell by cell (Section 6). `snake_rl.analysis.report` reads the run directories under `runs/` (per seed:
`config.yaml`, `metrics.csv`, `final_eval.json`) and writes the generated tables and figures to `runs/report/`;
`snake_rl.analysis.report_tables` computes the statistics the report's tables quote (paired differences on the same
seeds, the comparison with BFS on the same boards, the overestimation gap, the behaviour statistics, the tabular
sweep). No number of the report is typed by hand; these files are the numbers.

**Training logs**: `runs/` is not in the repository (about 1 GB with checkpoints). Three ways to get the logs into
this session: (1) upload `runs_logs.zip`, made with `python scripts/export_logs.py --include-behaviour` on the machine
that trained, when the cell of Section 2 asks for it; (2) set `LOGS_URL` in that cell to a link to the zip (for
example a GitHub release asset); (3) run the notebook inside a checkout that already has `runs/`. The zip holds the
per-run logs, the reference-player evaluations, the saved behaviour analyses (Table 9, Figures 11 to 13) and the
screenshot of Figure 1; it carries no weights.

### 1. Repository and Imports

In [ ]:
# Clone the repository and install the package when running on Google Colab; a local checkout is used as it is
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Cureeeeeeee/Value-Based-Deep-Reinforcement-Learning-on-Snake.git"

if not os.path.isdir("snake_rl"):
    if not os.path.isdir("snake-rl"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "snake-rl"], check=True)
    os.chdir("snake-rl")
if IN_COLAB:
    %pip install -q -e .
print("working directory:", os.getcwd())

In [ ]:
import glob
import json
import urllib.request
import zipfile

import numpy as np
import torch
from IPython.display import Image, Markdown, display

from snake_rl.analysis import report_tables as rt

REPORT_MD = "docs/PartE-final-report.md"   # the report source: captions, figure files and the tables to check against
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"python {sys.version.split()[0]}  torch {torch.__version__}  numpy {np.__version__}  device {DEVICE}")

### 2. Training Logs

In [ ]:
# Training logs: the local runs/ directory, or runs_logs.zip (scripts/export_logs.py) from LOGS_URL or the Colab upload dialog
LOGS_URL = ""               # link to runs_logs.zip; empty: upload it on Colab, or use the local runs/
LOGS_ZIP = "runs_logs.zip"


def experiment_dirs():
    return sorted(d for d in glob.glob("runs/*/") if glob.glob(os.path.join(d, "*", "final_eval.json")))


if not experiment_dirs():
    if LOGS_URL:
        urllib.request.urlretrieve(LOGS_URL, LOGS_ZIP)
    elif IN_COLAB and not os.path.exists(LOGS_ZIP):
        from google.colab import files

        LOGS_ZIP = next(iter(files.upload()))
    elif not os.path.exists(LOGS_ZIP):
        raise FileNotFoundError("no runs/ and no runs_logs.zip: set LOGS_URL or copy the zip next to this notebook")
    with zipfile.ZipFile(LOGS_ZIP) as zf:
        zf.extractall(".")  # entries are runs/<experiment>/<seed>/<file>

print(len(experiment_dirs()), "experiment directories with finished runs under runs/")

### 3. Experiment Matrix

The 64 configurations of the study are defined in `snake_rl/analysis/matrix.py`; the status table lists the finished
seeds of every row and checks each run's recorded `config.yaml` against the row that launched it.

In [ ]:
# Finished seeds per configuration, and every run's config.yaml checked against the matrix
!{sys.executable} -m snake_rl.analysis.matrix --status

### 4. Building the Report Outputs

#### 4.1 Generated tables and figures

In [ ]:
# Every generated table and figure (runs/report/); items whose experiments are missing are skipped and listed in runs/report/index.md
!{sys.executable} -m snake_rl.analysis.report --runs-dir runs --out-dir runs/report

#### 4.2 Composite figures

In [ ]:
# Figures 2, 9, 11 and 12 of the report: score distributions and death causes from the run directories, heat-maps from the saved behaviour analyses
sys.path.insert(0, "scripts")
from report_extra_figures import PANELS

needed = sorted({name for _, _, panels in PANELS for _, name in panels})
missing = [name for name in needed if not os.path.isdir(os.path.join("runs/behaviour", name))]
if missing:
    print("behaviour analyses missing under runs/behaviour:", ", ".join(missing))
    print("export the logs with --include-behaviour, or run snake_rl.analysis.behaviour for these players first")
else:
    !{sys.executable} scripts/report_extra_figures.py --out-dir runs/report/figures

#### 4.3 The statistics of the report's tables

In [ ]:
# Tables 3 to 9 and C1 of the report from the run directories; a table is skipped when its experiments are missing
tables, skipped = rt.build("runs")
for name in rt.TABLES:
    print(f"{name}: {'computed' if name in tables else 'skipped (' + skipped[name] + ')'}")

#### 4.4 The interface screenshot

In [ ]:
# Figure 1 is a screenshot of the GUI (scripts/gui_screenshots.py); rendered here when the two runs it plays are present with their weights
FIGURE1 = "runs/figures/partD_gui_agent.png"
if os.path.exists(FIGURE1):
    print("present:", FIGURE1)
elif all(os.path.exists(p) for p in ("runs/dqn_A_W10/0/model.pt", "runs/tabular_q_W10/0/q_table.npz")):
    !{sys.executable} scripts/gui_screenshots.py --out-dir runs/figures
else:
    print("not rendered: the screenshot needs the weights of runs/dqn_A_W10/0 and runs/tabular_q_W10/0")

### 5. The Report, Table by Table and Figure by Figure

Each item below is one table or figure of the report, with the report's caption. Tables 1 and 2 are text (the
training settings and the summary of the answers) and are shown as the report source has them; every other table is
computed from the logs in Section 4.3, and the figures are the files that Section 4 wrote.

In [ ]:
# Captions, figure files and source tables of the report, and the two display helpers
captions = rt.report_captions(REPORT_MD)
figure_files = rt.report_figures(REPORT_MD)
source_tables = rt.report_tables(REPORT_MD)


def show_table(name, markdown=None):
    """A computed table (``markdown``) or, without one, the table of the report source; with the report's caption."""
    if markdown is None:
        if name not in tables and name in skipped:
            print(f"{name}: not computed ({skipped[name]})")
            return
        rows = source_tables[name]
        markdown = rt.markdown_table(rows[0], rows[1:])
    display(Markdown(f"**{name}.** {captions[name]}\n\n{markdown}"))


def show_figure(name, files=None):
    """The file(s) of a figure of the report with the report's caption."""
    files = figure_files[name] if files is None else files
    for f in files:
        if os.path.exists(f):
            display(Image(filename=f))
        else:
            print(f"{name}: missing {f}")
    display(Markdown(f"**{name}.** {captions[name]}"))

#### 5.1 The interface (Section 3.6): Figure 1

In [ ]:
show_figure("Figure 1")

#### 5.2 Training settings (Section 4): Table 1

In [ ]:
show_table("Table 1")

#### 5.3 The questions and the answers (Section 6): Table 2

In [ ]:
show_table("Table 2")

#### 5.4 The scale and the tabular reference (Section 6.1): Table 3 and Figure 2

In [ ]:
show_table("Table 3", tables.get("Table 3"))
show_figure("Figure 2")

#### 5.5 State representation (Section 6.2): Table 4 and Figure 3

In [ ]:
show_table("Table 4", tables.get("Table 4"))
show_figure("Figure 3")

#### 5.6 Discount factor (Section 6.3): Table 5 and Figure 4

In [ ]:
show_table("Table 5", tables.get("Table 5"))
show_figure("Figure 4")

#### 5.7 The stabilisers (Section 6.4): Table 6, Figures 5 and 6

In [ ]:
show_table("Table 6", tables.get("Table 6"))
show_figure("Figure 5")
show_figure("Figure 6")

#### 5.8 Double DQN, Dueling and prioritised replay (Section 6.5): Table 7 and Figure 7

In [ ]:
show_table("Table 7", tables.get("Table 7"))
show_figure("Figure 7")

#### 5.9 Reward design (Section 6.6): Table 8, Figures 8 and 9

In [ ]:
show_table("Table 8", tables.get("Table 8"))
show_figure("Figure 8")
show_figure("Figure 9")

#### 5.10 Reward hacking, the probe (Section 6.7): Figure 10

In [ ]:
show_figure("Figure 10")

#### 5.11 Behaviour (Section 6.8): Table 9, Figures 11 to 13

In [ ]:
show_table("Table 9", tables.get("Table 9"))
show_figure("Figure 11")
show_figure("Figure 12")
show_figure("Figure 13")

#### 5.12 Every configuration (Appendix B): Tables B1 and B2

In [ ]:
# The columns of the report from the generated summary tables of both boards
for name, W in (("Table B1", 8), ("Table B2", 10)):
    show_table(name, rt.appendix_table(f"runs/report/tables/summary_W{W}.md"))

#### 5.13 The tabular sweep (Appendix C): Table C1

In [ ]:
show_table("Table C1", tables.get("Table C1"))

### 6. Check Against the Report

Every computed table is compared with the table in the report source cell by cell, and every figure file of the
report must exist.

In [ ]:
# Computed tables against the report source, and the figure files of the report
problems = 0
for name in rt.TABLES:
    if name not in tables:
        print(f"{name}: not computed ({skipped[name]})")
        problems += 1
        continue
    diffs = rt.compare_with_report(name, tables[name], REPORT_MD)
    problems += len(diffs)
    print(f"{name}: {'identical to the report' if not diffs else str(len(diffs)) + ' differences'}")
    for d in diffs:
        print("  " + d)
for name, files in figure_files.items():
    missing = [f for f in files if not os.path.exists(f)]
    if missing:
        problems += 1
        print(f"{name}: missing {', '.join(missing)}")
    else:
        print(f"{name}: {', '.join(files)}")
print("RESULT:", "every table and figure of the report is reproduced" if problems == 0 else f"{problems} problem(s)")

### 7. Behaviour Analysis of One Run (needs weights)

`snake_rl.analysis.behaviour` plays 100 evaluation episodes with a trained network and writes the visit heat-map, the
loop / shaping statistics per episode and GUI-replayable recordings of the longest loops
(`python -m snake_rl.gui.app --replay <file>`). The saved analyses behind Table 9 and Figures 11 to 13 were made this
way; running one needs a `model.pt`, which the logs zip does not carry: train a run
(`python -m snake_rl.train --config configs/dqn_B_8x8.yaml --seed 0 --set agent.gamma=0.99 --exp dqn_B8_W8_gamma099`)
or copy a run directory here.

In [ ]:
# Behaviour analysis of one run: 100 greedy episodes, heat-map, loop statistics and recordings of the longest loops
RUN_DIR = "runs/dqn_B8_W8_gamma099/0"   # any run directory with model.pt
OUT_DIR = "runs/behaviour/notebook"

if os.path.exists(os.path.join(RUN_DIR, "model.pt")):
    !{sys.executable} -m snake_rl.analysis.behaviour --player {RUN_DIR} --checkpoint final --device {DEVICE} --out-dir {OUT_DIR}
    display(Image(filename=os.path.join(OUT_DIR, "heatmap.png")))
    display(Image(filename=os.path.join(OUT_DIR, "last_steps.png")))
    with open(os.path.join(OUT_DIR, "summary.json"), encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:2000])
else:
    print("no weights at", RUN_DIR, "- the logs zip carries none; see the note above")